# Práctica 7: Agrupamiento de Datos (Clustering - K-Means)

**Objetivo:** Crear un modelo de agrupamiento utilizando el algoritmo **K-Means** para identificar diferentes tipos de situaciones de tiro en la Premier League (temporada 2024-2025).

### Variables seleccionadas para el clustering:
- `shot_x`, `shot_y`: Coordenadas del tiro.
- `xg`: Probabilidad estadística previa de gol (Expected Goals).
- `isHome`: Si el equipo es local o visitante.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Cargar datos
df = pd.read_csv('cleaned_epl_shots.csv')

print(f"Dimensiones del dataset: {df.shape}")

## Preprocesamiento de Datos

K-Means se basa en distancias euclidianas, por lo que el escalado de datos es fundamental.

In [ ]:
# Selección de características
features = ['shot_x', 'shot_y', 'xg', 'isHome']
X = df[features].copy()
X['isHome'] = X['isHome'].astype(int)

# Escalado de datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Datos escalados y listos para clustering.")

## Determinación del número óptimo de Clústeres (K)

Utilizaremos el **Método del Codo (Elbow Method)** para encontrar el valor de $K$ ideal.

In [ ]:
wcss = [] # Within-Cluster Sum of Square

for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10,6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Método del Codo')
plt.xlabel('Número de Clústeres (K)')
plt.ylabel('WCSS')
plt.show()

## Aplicación de K-Means con el K óptimo

Basado en el gráfico anterior, seleccionaremos un valor adecuado (por ejemplo, $K=4$ o $K=5$).

In [ ]:
k_optimo = 4
kmeans = KMeans(n_clusters=k_optimo, init='k-means++', random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print(f"Distribución de los clústeres:\n{df['cluster'].value_counts()}")

## Visualización de los Clústeres

Visualizaremos cómo se agrupan los tiros en el campo de juego (`shot_x`, `shot_y`).

In [ ]:
plt.figure(figsize=(12,8))
sns.scatterplot(data=df, x='shot_x', y='shot_y', hue='cluster', palette='viridis', alpha=0.6)
plt.title('Agrupamiento de Tiros (K-Means)')
plt.xlabel('Coordenada X (Distancia a la meta)')
plt.ylabel('Coordenada Y (Ancho del campo)')
plt.legend(title='Clúster')
plt.show()

## Interpretación de los Clústeres

Analizamos los promedios de cada grupo para entender qué representan.

In [ ]:
cluster_analysis = df.groupby('cluster')[['shot_x', 'shot_y', 'xg', 'isHome']].mean()
print("Análisis promedio por clúster:")
print(cluster_analysis)

## Conclusiones

1. **Identificación de Patrones:** El clustering nos permite ver grupos de tiros: tiros lejanos, tiros en el área chica, tiros desde los costados, etc.
2. **Peligrosidad:** Se puede observar que ciertos clústeres tienen un promedio de `xg` (Expected Goals) significativamente mayor, lo que indica que representan situaciones de gol más claras.
3. **Uso en Estrategia:** Este tipo de análisis ayuda a entender las zonas preferidas de los equipos para atacar y la efectividad de sus disparos según la ubicación.